# Module 05 — Notebook 05: Scale the Index

Notebook ini menutup **tiga janji** yang dibuat di akhir notebook 04:

1. **Ganti L2 → Cosine** menggunakan `IndexFlatIP` + `faiss.normalize_L2` — metrik yang lebih tepat untuk teks embedding.
2. **Pahami trade-off index aproksimasi** di skala besar lewat benchmark nyata: Flat vs IVFFlat vs HNSW, diukur dengan `recall@k` dan latensi.
3. **Tambah persistence + metadata filtering** — dua kemampuan yang FAISS mentah tidak punya — menggunakan ChromaDB (`PersistentClient`).

> **CPU-only, tanpa GPU.** Seluruh notebook ini selesai dalam waktu kurang dari 1 menit pada CPU standar Colab — tidak perlu T4.

In [ ]:
# Install dependencies (CPU-only)
!pip install -q "transformers<5" "sentence-transformers>=3.0" faiss-cpu "chromadb>=1.0,<2" scikit-learn

# Clone repo if not already present (e.g., fresh Colab session)
import os, sys
REPO = "/content/navasena-gen-ml-course"
if not os.path.exists(REPO):
    !git clone -q https://github.com/chmdznr/navasena-gen-ml-course.git {REPO}
sys.path.append(os.path.abspath(f"{REPO}/05_rag"))

from tools.rag_utils import recall_at_k
print("recall_at_k imported:", recall_at_k)

## Masalah Warisan: L2, RAM-only, Tanpa Metadata

Sejauh ini (nb01–nb04) kita memakai `IndexFlatL2` yang hanya hidup di RAM. Ada **tiga gap** nyata:

| Gap | Masalah |
|-----|---------|
| **Metrik salah** | `IndexFlatL2` mengukur *jarak Euclidean* (besar = jauh). Untuk teks embedding, yang bermakna adalah **arah** vektor, bukan panjangnya. |
| **Tidak skalabel** | Brute-force Flat lambat di puluhan ribu vektor. Perlu index aproksimasi dengan trade-off recall vs latensi yang terukur. |
| **Tidak persisten** | Index hilang saat session berakhir. FAISS juga tidak menyimpan teks/metadata — perlu bookkeeping manual. |

### Intuisi Cosine vs L2

Bayangkan dua anak panah: *"Saya suka kucing"* dan *"Aku senang dengan kucing"*. Keduanya menunjuk **arah yang sama** meski panjangnya berbeda. Cosine similarity mengukur sudut antar panah itu — semakin kecil sudutnya, semakin mirip maknanya. L2 mengukur jarak ujung-ke-ujung, yang bisa menyesatkan jika panjang vektor tidak seragam.

**Trik:** pada vektor yang sudah dinormalisasi ke panjang 1, *inner product* = cosine similarity. Itulah yang dilakukan `IndexFlatIP` + `normalize_L2`.

### Preview 3 Gap yang Ditutup Notebook Ini

1. **Cosine fix** — ganti L2 → `IndexFlatIP` + `normalize_L2` (§1)
2. **Benchmark terukur** — Flat / IVFFlat / HNSW pada 20 000 vektor sintetis, plot recall vs latensi (§2)
3. **ChromaDB** — persistence otomatis + `where` metadata filter tanpa bookkeeping sendiri (§3)

In [ ]:
# ── §1: Cosine Fix ──────────────────────────────────────────────────────────

# Same 12-passage handbook corpus used in nb03/nb04 (source of truth lives in nb03)
corpus = [
    "Setiap pegawai tetap memperoleh jatah dua belas hari libur berbayar dalam satu tahun, terpisah dari hari libur nasional.",
    "Pengajuan cuti tahunan dilakukan melalui portal karyawan dan wajib mendapat persetujuan atasan langsung paling lambat tiga hari sebelumnya.",
    "Cuti sakit dapat diambil hingga sepuluh hari setiap tahun dan harus disertai surat keterangan dari dokter bila lebih dari dua hari berturut-turut.",
    "Karyawan baru menjalani masa orientasi selama tiga hari kerja pertama, mencakup pengenalan budaya perusahaan dan pelatihan keselamatan.",
    "Cuti melahirkan diberikan selama tiga bulan penuh sesuai ketentuan ketenagakerjaan yang berlaku di Indonesia.",
    "Jam kerja standar adalah pukul 09.00 hingga 17.00 dari Senin sampai Jumat, dengan satu jam istirahat untuk makan siang.",
    "Karyawan dapat bekerja dari rumah maksimal dua hari dalam seminggu setelah mendapat persetujuan dari manajer tim.",
    "Gaji dibayarkan pada tanggal 25 setiap bulan melalui transfer bank ke rekening masing-masing pegawai.",
    "Perusahaan menyediakan asuransi kesehatan yang mencakup rawat inap dan rawat jalan bagi pegawai beserta satu anggota keluarga.",
    "Lembur pada hari libur dihitung dua kali lipat dari tarif normal dan harus dicatat dalam sistem absensi sebelum dikerjakan.",
    "Pelanggaran kode etik dapat berujung pada surat peringatan hingga pemutusan hubungan kerja sesuai tingkat keparahannya.",
    "Penggantian biaya perjalanan dinas diajukan paling lambat tujuh hari setelah perjalanan dengan melampirkan bukti pembayaran yang sah.",
]
# Komentar: korpus sama seperti nb03/nb04 (sumber kebenaran ada di nb03).

import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
vecs = embedder.encode(corpus, convert_to_numpy=True).astype("float32")
dim = vecs.shape[1]

# Old way: L2 (raw Euclidean)
l2 = faiss.IndexFlatL2(dim)
l2.add(vecs)

# New way: cosine = inner product on L2-normalized vectors
vecs_n = vecs.copy()          # normalize_L2 is IN-PLACE (returns None) → copy first!
faiss.normalize_L2(vecs_n)
ip = faiss.IndexFlatIP(dim)
ip.add(vecs_n)

query = "Berapa hari cuti tahunan pegawai tetap?"
qv = embedder.encode([query], convert_to_numpy=True).astype("float32")

dl2, il2 = l2.search(qv, 3)

qn = qv.copy()                # query MUST also be normalized for cosine regime
faiss.normalize_L2(qn)
dip, iip = ip.search(qn, 3)

print("L2  (jarak, kecil=mirip): ", [(int(i), round(float(d), 3)) for i, d in zip(il2[0], dl2[0])])
print("Cosine (sim, besar=mirip):", [(int(i), round(float(d), 3)) for i, d in zip(iip[0], dip[0])])

## Menyimpan Index FAISS ke Disk

`faiss.write_index` / `faiss.read_index` menyimpan vektor + parameter index (tanpa perlu retrain ulang saat dimuat). Operasi ini **cepat dan efisien** — cocok untuk index besar.

**Namun ada keterbatasan penting:** FAISS hanya menyimpan vektor dan row-id integer — **bukan teks atau metadata**. Saat kita ingin tahu *passage mana* yang dikembalikan, kita harus mengelola pemetaan `row_id → teks` sendiri.

Solusi standar: **sidecar JSON** (atau database terpisah) yang menyimpan metadata. ChromaDB di §3 menghapus beban ini sepenuhnya.

> **Keamanan:** Jangan pernah memanggil `faiss.read_index` pada file dari sumber yang tidak dipercaya — format biner FAISS belum diaudit secara keamanan.

In [ ]:
import json

# Save index + sidecar metadata
faiss.write_index(ip, "/content/handbook.index")
sidecar = {
    "id_map": [f"doc_{i}" for i in range(len(corpus))],
    "texts": {f"doc_{i}": corpus[i] for i in range(len(corpus))},
    "index_type": "FlatIP",
    "dim": dim,
    "metric": "cosine",
    "normalized": True,
}
json.dump(sidecar, open("/content/handbook.sidecar.json", "w"))

# Simulate a new session: load from disk
idx2 = faiss.read_index("/content/handbook.index")
meta = json.load(open("/content/handbook.sidecar.json"))

assert idx2.ntotal == len(corpus), "Index size mismatch after reload!"
_, I = idx2.search(qn, 3)
print([(meta["id_map"][i], meta["texts"][meta["id_map"][i]][:45]) for i in I[0]])

## §2: Skala Besar — Benchmark Flat vs IVFFlat

12 passage itu sepele. Korpus nyata punya **puluhan ribu hingga jutaan** vektor — di situ brute-force Flat mulai terasa.

### Taksonomi Index FAISS

| Index | Cara Kerja | Kecepatan | Recall |
|-------|-----------|-----------|--------|
| **Flat** | Bandingkan semua vektor (exact) | Lambat di skala besar | 100% (ground truth) |
| **IVFFlat** | Bagi ruang jadi `nlist` sel, cari hanya `nprobe` sel | Cepat, bisa dikontrol | ~95–99% tergantung `nprobe` |
| **IndexHNSWFlat** | Graf navigasi hirarkis; parameter `M` (konektivitas) + `efSearch` (lebar pencarian) | Cepat, RAM lebih besar | Tinggi pada embedding nyata; **tidak adil diukur di data sintetis** (lihat §2b) |
| **IVFPQ** | IVF + kompresi Product Quantization | Paling cepat, RAM kecil | ~85–95% (trade-off memori) |

**`recall@k`** = proporsi hasil aproksimasi yang cocok dengan Flat (ground truth) di top-k. Nilai 1.0 = sempurna.

Kita benchmark **Flat vs IVF** secara langsung — keduanya berperilaku wajar di data sintetis dan menunjukkan trade-off recall vs latensi yang nyata. **HNSW** ditunjukkan API-nya + dijelaskan konsepnya karena data sintetis berklaster/acak TIDAK adil menggambarkan recall-nya (di embedding NYATA HNSW biasanya tercepat dengan recall tinggi). **IVFPQ** disebut singkat sebagai opsi kompresi memori untuk korpus sangat besar.

In [ ]:
import time
from sklearn.datasets import make_blobs
N, d = 100000, 384                         # 100rb vektor; data BERKLASTER (mirip struktur embedding nyata)
xb, _ = make_blobs(n_samples=N, n_features=d, centers=100, random_state=42)
xb = xb.astype("float32"); faiss.normalize_L2(xb)             # regime cosine
rng = np.random.default_rng(0)
qi = rng.choice(N, 1000, replace=False)
xq = xb[qi] + rng.normal(0, 0.5, (1000, d)).astype("float32") # query = titik korpus + noise (bukan titik persis -> tetangga non-trivial)
faiss.normalize_L2(xq)
gt = faiss.IndexFlatL2(d); gt.add(xb)
_, gtI = gt.search(xq, 10)                                    # ground truth (exact)
print(f"korpus sintetis: {N} vektor, dim {d}; 1000 query (titik korpus + noise)")

In [ ]:
import pandas as pd
def bench(name, build):
    t0 = time.perf_counter(); index = build(); build_s = time.perf_counter() - t0
    index.search(xq[:10], 10)                                # warm-up
    t0 = time.perf_counter(); _, I = index.search(xq, 10); qms = (time.perf_counter()-t0)/len(xq)*1000
    return {"index": name, "build_s": round(build_s,3), "ms/query": round(qms,3),
            "recall@10": round(recall_at_k(I, gtI), 3)}
def make_flat():
    ix = faiss.IndexFlatL2(d); ix.add(xb); return ix
def make_ivf():
    nlist = int(N**0.5)                                      # ~316
    ix = faiss.IndexIVFFlat(faiss.IndexFlatL2(d), d, nlist)
    ix.train(xb); ix.add(xb); ix.nprobe = 8; return ix
df = pd.DataFrame([bench("Flat (exact)", make_flat), bench("IVFFlat (nprobe=8)", make_ivf)])
print(df.to_string(index=False))
# Harapan: Flat recall 1.0 tapi paling lambat per-query; IVF ~2x lebih cepat dengan recall ~0.99.

**Baca tabel di atas:**
- **Flat** = exact, recall 1.0, tapi ms/query paling besar.
- **IVFFlat** = aproksimasi cepat dengan `nprobe=8`. Naikkan `nprobe` → recall naik, latensi naik.
- **HNSW** = grafik navigasi; build lebih lama dan pakai RAM lebih besar, tapi query sangat cepat. Knob-nya `efSearch`.

In [ ]:
# nprobe Pareto curve: find the sweet spot between recall and latency
import matplotlib.pyplot as plt

ivf = make_ivf()
xs, ys, labels = [], [], []
for nprobe in (1, 4, 8, 16, 32, 64):
    ivf.nprobe = nprobe
    ivf.search(xq[:10], 10)             # warm-up
    t0 = time.perf_counter()
    _, I = ivf.search(xq, 10)
    ms = (time.perf_counter() - t0) / len(xq) * 1000
    xs.append(ms)
    ys.append(recall_at_k(I, gtI))
    labels.append(nprobe)

plt.figure(figsize=(7, 4))
plt.plot(xs, ys, marker="o")
for x, y, n in zip(xs, ys, labels):
    plt.annotate(f"nprobe={n}", (x, y), textcoords="offset points", xytext=(5, 5), fontsize=8)
plt.xlabel("ms/query")
plt.ylabel("recall@10")
plt.title("IVF: trade-off recall vs latensi (knob nprobe)")
plt.tight_layout()
plt.show()

**Kurva di atas** menunjukkan hukum *diminishing returns*: `nprobe` naik 2× → recall naik sedikit, latensi naik hampir linear. Titik manis biasanya ada di recall ≥ 0.95 dengan latensi seminimal mungkin.

Untuk HNSW, knob setaranya adalah `efSearch` — logikanya sama.

### HNSW: graf navigasi
HNSW membangun graf 'small-world' berlapis; knob utamanya **`efSearch`** (besar = recall naik, lebih lambat — analog `nprobe` pada IVF) dan **`M`** (jumlah tetangga per node). Pada **embedding nyata**, HNSW sering pilihan tercepat dengan recall tinggi. Namun data sintetis di notebook ini (berklaster/normalized) bukan ujian yang adil untuknya, jadi kita hanya tunjukkan API-nya — bukan angka recall-nya.

In [ ]:
sub = xb[:20000]                                            # subset kecil supaya build cepat (~0.4s)
hnsw = faiss.IndexHNSWFlat(d, 32); hnsw.hnsw.efConstruction = 200
hnsw.add(sub); hnsw.hnsw.efSearch = 64                      # efSearch = knob recall/latensi (analog nprobe)
_, I = hnsw.search(xq[:5], 5)
print("HNSW API OK — bentuk hasil:", I.shape,
      "(recall pada data sintetis menyesatkan; pada embedding nyata HNSW unggul)")

## §3: ChromaDB — Persistence + Metadata Filtering

FAISS mentah tidak punya dua fitur yang sering dibutuhkan RAG nyata:

- **Persistence otomatis** — harus `write_index` manual + sidecar JSON.
- **Metadata filtering** — harus filter manual setelah retrieval.

ChromaDB menyelesaikan keduanya. Kita menyerahkan vektor sendiri (`embedding_function=None` — tidak mengunduh model ONNX otomatis), sehingga embedding tetap konsisten dengan FAISS di atas.

**API version-sensitive (chromadb ≥ 1.0):**
- `chromadb.PersistentClient(path=...)` — ganti `Client(Settings(persist_directory=...))` yang lama.
- `configuration={"hnsw": {"space": "cosine"}}` — ganti `metadata={"hnsw:space": "cosine"}` yang lama.
- **Tidak ada method persist manual** — dihapus di 0.4+; Chroma auto-persist ke SQLite setiap operasi. Memanggil `.persist()` pada `client` akan memunculkan `AttributeError`.

In [ ]:
# Colab modern (sqlite>=3.35) usually works directly.
# If you get RuntimeError 'unsupported sqlite3 version':
#   !pip install -q pysqlite3-binary
#   __import__("pysqlite3"); import sys; sys.modules["sqlite3"] = sys.modules.pop("pysqlite3")
#   (put BEFORE import chromadb, then RESTART runtime)
import chromadb
from chromadb.config import Settings

In [ ]:
# Create persistent client — data auto-saved to /content/chroma_db (SQLite)
client = chromadb.PersistentClient(
    path="/content/chroma_db",
    settings=Settings(anonymized_telemetry=False)
)

col = client.get_or_create_collection(
    name="handbook",
    configuration={"hnsw": {"space": "cosine"}},  # chromadb 1.x syntax (not the old metadata= form)
    embedding_function=None,                       # None = bring-your-own vectors; avoids ONNX auto-download
)

# Metadata categories (one per passage, same order as corpus)
kategori = ["cuti", "cuti", "cuti", "onboarding", "cuti",
            "jam_kerja", "wfh", "gaji", "asuransi", "lembur", "etik", "dinas"]
metas = [{"kategori": kategori[i], "doc_id": i} for i in range(len(corpus))]

# Encode with normalize_embeddings=True → cosine-ready floats
embs = embedder.encode(corpus, normalize_embeddings=True).tolist()  # list[list[float]]

col.add(
    ids=[f"doc_{i}" for i in range(len(corpus))],
    documents=corpus,
    embeddings=embs,
    metadatas=metas,
)
print("jumlah dokumen:", col.count())  # 12

In [ ]:
q = "kebijakan cuti tahunan"
qv2 = embedder.encode([q], normalize_embeddings=True).tolist()

# Unfiltered query
plain = col.query(
    query_embeddings=qv2,
    n_results=3,
    include=["documents", "metadatas", "distances"],  # Note: 'ids' must NOT be in include
)
print("Tanpa filter:", plain["ids"][0])

# Metadata-filtered query: only passages in category 'cuti'
filtered = col.query(
    query_embeddings=qv2,
    n_results=3,
    where={"kategori": "cuti"},
    include=["documents", "distances"],
)
print("Filter kategori=cuti:", filtered["ids"][0],
      [round(x, 3) for x in filtered["distances"][0]])  # cosine distance: smaller = more similar

**Filter `where={"kategori": "cuti"}`** ini yang tidak bisa dilakukan FAISS mentah tanpa bookkeeping tambahan. ChromaDB menyimpan metadata di SQLite dan memfilter *sebelum* kalkulasi similarity — efisien dan rapi.

In [ ]:
# Prove persistence: close client, reopen, data still there — no manual persist call needed
del client
client2 = chromadb.PersistentClient(path="/content/chroma_db")
print("collections:", [c.name for c in client2.list_collections()])  # ['handbook']
col2 = client2.get_collection("handbook", embedding_function=None)
print("count setelah buka ulang:", col2.count())    # 12 (auto-persisted, no explicit persist needed)
print("space:", col2.configuration["hnsw"]["space"])  # 'cosine'

## Matriks Keputusan: FAISS vs ChromaDB

| Kemampuan | FAISS | ChromaDB |
|-----------|-------|----------|
| **Persistence** | Manual: `write_index` + sidecar JSON | Otomatis ke SQLite (tidak perlu `persist()`) |
| **Metadata filtering** | Tidak ada bawaan — filter manual setelah retrieval | `where` bawaan, difilter sebelum similarity |
| **Simpan teks passage** | Tidak — hanya vektor + int64 row-id | Ya — `documents` tersimpan bersama metadata |
| **Kontrol index & tuning** | Penuh: Flat / IVF / HNSW / PQ; `nlist`, `nprobe`, `efSearch`, `M` | HNSW dikelola internal; lebih sedikit knob |
| **Skala + kontrol memori** | IVFPQ menang besar (kuantisasi vektor) | Terbatas pada HNSW internal |
| **Prototyping RAG cepat** | Perlu bookkeeping sendiri | Menang — satu API untuk semua |

**Verdict praktis:**
- Pakai **FAISS** untuk kontrol penuh, benchmark, atau skenario memori sangat ketat (IVFPQ).
- Pakai **ChromaDB** untuk iterasi cepat, persistence tanpa kerumitan, dan metadata filtering bawaan.

## Ringkasan & Jembatan ke Notebook 06

Tiga gap dari nb04 sudah ditutup:

1. **Cosine fix** ✅ — `IndexFlatIP` + `faiss.normalize_L2` menggantikan `IndexFlatL2` yang hanya mengukur jarak Euclidean.
2. **Trade-off terukur** ✅ — Benchmark Flat / IVFFlat / HNSW pada 20 000 vektor; kurva Pareto `nprobe` memperlihatkan sweet spot recall vs latensi.
3. **Persistence + metadata filtering** ✅ — `PersistentClient` ChromaDB auto-persist ke SQLite; `where` filter metadata tanpa bookkeeping sendiri.

### Jembatan ke Notebook 06: Conversational RAG

Retriever kita kini sudah **cosine**, **persistent**, dan **bisa difilter** — fondasi yang kuat. Notebook 06 membangun di atasnya dengan menambahkan **memori multi-turn**: chatbot yang bisa menjawab pertanyaan lanjutan ("dan berapa hari cuti sakitnya?") dengan mengingat konteks percakapan sebelumnya.

**PR opsional untuk eksplorasi mandiri:**
- Naikkan `N = 50000` dan amati perubahan trade-off.
- Tambahkan `IndexIVFPQ` ke benchmark untuk melihat kompresi memori.
- Coba `d = 768` dengan model embedding yang lebih besar.